In [1]:
import os
import time

from bng_simulator.utils.services_utils import send_request
from bng_simulator.utils.io_dict_utils import(
    save_yaml,
    round_dict_values,
    build_tree_from_dict,
    convert_tree_into_proper_dict
)

In [2]:
# Get config, scenario , infos
scenario_infos = send_request("get_sim_config")

In [5]:
# Print the vehicles in the environment
for v, v_info in scenario_infos["vehicles"].items():
    _infos = {_k : _v for _k, _v in v_info.items() if _k not in ["scenario_args", "sensors"]}
    print(f"Vehicle {v} :")
    for _k, _v in _infos.items():
        print(f"\t{_k} : {_v}")

Vehicle ego :
	controllers : {'LowLevelController': {'calibration': {'commandTimeout': 1.0, 'controlMode': 'auto', 'gtStateSendRate': 0.01, 'modelName': 'wheel_speed_wit_integ_v1.json'}, 'controllerRate': 0.005, 'controllerType': 'nn_mpc', 'drivetrain': {'mode': 2, 'startGear': 2}, 'gtStateName': 'gtstate', 'listenIp': '127.0.0.1', 'listenPort': 64257, 'sendIp': '127.0.0.1', 'sendPort': 64258}}
	license : EGO
	model : utv
	partConfig : /vehicles/utv/wild.pc


In [6]:
# Print the part numbers, used as unique identifiers for the vehicle components
for v_name, v_part in scenario_infos["vehicles_part"].items():
    print(f"Vehicle {v_name} :  {v_part}")

Vehicle ego :  utv_wild


In [7]:
# Let's enumerate all gear options and their characteristics
def extract_gear_infos(vehicle_name = None):
    """
    Go through all gears and extract their characteristics
    """
    in_args = {}
    in_args_set = {}
    if vehicle_name is not None:
        in_args["vehicle_name"] = vehicle_name
        in_args_set["vehicle_name"] = vehicle_name
    # Let's first get current infos
    gear_infos = send_request("get_gearbox_info", in_args)
    min_gear = int(gear_infos["minGearIndex"])
    max_gear = int(gear_infos["maxGearIndex"])
    # TODO: The min is actually negative of the actual min cause of abs.
    print(f"Min/max gear: {min_gear}/{max_gear}")
    
    # A function to parse the gear infos
    def parse_gear_infos(gear_infos):
        curr_gear = int(gear_infos["gearIndex"])
        gear_mode_index = gear_infos["gearModeIndex"]
        gear_ratio = gear_infos["gearRatio"]
        gear_name = gear_infos["gearName"]
        gear_mode = gear_infos["mode"]
        return gear_name, \
            {"index": curr_gear, "mode_index": gear_mode_index, 
             "gear_ratio": gear_ratio, "mode": gear_mode}
    
    # Store the different gear ratio
    mapping_info = {}
    gear_name, gear_info = parse_gear_infos(gear_infos)
    print(f"Gear {gear_name} - {gear_info}")
    initial_gear_name = gear_name
    for i in range(-1, max_gear + 3): # Just extra for the sake of it
        in_args_set["gear_index"] = i
        send_request("set_gearbox_index", in_args_set)
        # Wait for a bit
        time.sleep(0.5)
        # Get the gear infos
        gear_infos = send_request("get_gearbox_info", in_args)
        gear_name, gear_info = parse_gear_infos(gear_infos)
        gear_info["indexCmd"] = i
        print(f"Gear {gear_name} - {gear_info}")
        if gear_name not in mapping_info:
            mapping_info[gear_name] = gear_info
    # Set the initial gear index
    initial_gear_index = mapping_info[initial_gear_name]["indexCmd"]
    in_args_set["gear_index"] = initial_gear_index
    send_request("set_gearbox_index", in_args_set)
    return mapping_info

In [8]:
# Extract the gear infos
gear_infos = extract_gear_infos()

Min/max gear: 1/1
Gear D - {'index': 1, 'mode_index': 0.78, 'gear_ratio': 3.5, 'mode': 'drive'}
Gear R - {'index': -1, 'mode_index': 0.4, 'gear_ratio': 3.5, 'mode': 'reverse', 'indexCmd': -1}
Gear N - {'index': 0, 'mode_index': 0.55, 'gear_ratio': 3.5, 'mode': 'neutral', 'indexCmd': 0}
Gear P - {'index': 0, 'mode_index': 0.0, 'gear_ratio': 3.5, 'mode': 'park', 'indexCmd': 1}
Gear D - {'index': 1, 'mode_index': 0.78, 'gear_ratio': 3.5, 'mode': 'drive', 'indexCmd': 2}
Gear N - {'index': 0, 'mode_index': 0.55, 'gear_ratio': 3.5, 'mode': 'neutral', 'indexCmd': 3}


In [9]:
gear_infos

{'R': {'index': -1,
  'mode_index': 0.4,
  'gear_ratio': 3.5,
  'mode': 'reverse',
  'indexCmd': -1},
 'N': {'index': 0,
  'mode_index': 0.55,
  'gear_ratio': 3.5,
  'mode': 'neutral',
  'indexCmd': 0},
 'P': {'index': 0,
  'mode_index': 0.0,
  'gear_ratio': 3.5,
  'mode': 'park',
  'indexCmd': 1},
 'D': {'index': 1,
  'mode_index': 0.78,
  'gear_ratio': 3.5,
  'mode': 'drive',
  'indexCmd': 2}}

In [10]:
# Useful functions to format the powertrain structure
def format_powertrain_structure(data, relevant_keys):
    """
    Format the powertrain structure.
    
    Args:
        data (Dict[str, Any]): The data.
        relevant_keys (Tuple[str]): The relevant keys.
        
    Returns:
        Dict[str, Any]: The tree-like structure.
    """
    roots, tree = build_tree_from_dict(data)
    res = {}
    for root in roots:
        res[root] = convert_tree_into_proper_dict(root, tree, data, relevant_keys)
    # # Let;s sort it
    # res = dict
    return round_dict_values(res)

powertrain_prop = send_request("get_powertrain_properties")
powertrain_infos = format_powertrain_structure(
    powertrain_prop,
    ['type', 'gearRatio', 'gearRatios', 'availableModes', 'diffTorqueSplit']
)

In [11]:
powertrain_infos

{'mainEngine': {'type': 'combustionEngine',
  'clutch': {'type': 'centrifugalClutch',
   'gearRatio': 1.0,
   'gearbox': {'type': 'cvtGearbox',
    'gearRatio': 3.5,
    'rangebox': {'type': 'rangeBox',
     'gearRatio': 2.23,
     'gearRatios': {0.0: 2.23, 1.0: 4.56},
     'availableModes': ['high', 'low'],
     'transfercase': {'type': 'differential',
      'gearRatio': 1.0,
      'availableModes': ['locked'],
      'diffTorqueSplit': 0.5,
      'differential_R': {'type': 'differential',
       'gearRatio': 3.82,
       'availableModes': ['locked'],
       'diffTorqueSplit': 0.5,
       'wheelaxleRL': {'type': 'shaft',
        'gearRatio': 1.0,
        'availableModes': ['connected'],
        'spindleRL': {'type': 'shaft',
         'gearRatio': 1.0,
         'availableModes': ['connected']}},
       'wheelaxleRR': {'type': 'shaft',
        'gearRatio': 1.0,
        'availableModes': ['connected'],
        'spindleRR': {'type': 'shaft',
         'gearRatio': 1.0,
         'availableMo

In [12]:
# Get controllers infos
controllers_infos = send_request("get_controller_infos")
controllers_infos = dict(sorted(controllers_infos.items()))

In [13]:
controllers_infos

{'4wd': '4wd',
 'doorLCoupler': 'advancedCouplerControl',
 'doorRCoupler': 'advancedCouplerControl',
 'frontBypass': 'bypassDampers',
 'frontLockControl': 'driveModes',
 'gauge': 'gauges/genericGauges',
 'gauges/customModules/combustionEngineData': 'gauges/customModules/combustionEngineData',
 'gauges/customModules/environmentData': 'gauges/customModules/environmentData',
 'gtState0': 'xlab/gtState',
 'rangeboxControl': 'driveModes',
 'rearBypass': 'bypassDampers',
 'singleAxisLever': 'propAnimation/singleAxisLever',
 'transfercaseControl': 'driveModes',
 'twoStepLaunch': 'twoStepLaunch',
 'utv_naviscreen_accessory': 'beamNavigator',
 'vehicleController': 'vehicleController/vehicleController',
 'xlabControllerManager': 'xlab/controller_manager'}

In [14]:
# Kinematics infos
kin_args = {
    "world_space": False
}
kin_props = send_request("get_vehicle_properties", kin_args)

In [15]:
# Handle engine infos
engine_infos = send_request("get_engine_infos")

In [ ]:
engine_infos

In [16]:
kin_props

{'cogPosDynamic': [0.23102211588411592, 0.6713089179247618, 0.564983069896698],
 'cogPosDynamicRel': [0.0020040414483422525,
  0.16404343382128148,
  0.6543882492539687],
 'cogPosStatic': [0.23093564148349643, 0.6603114607766536, 0.5540847220147496],
 'cogPosStaticRel': [0.2310040414483423,
  0.6740434338212815,
  0.2613882492539686],
 'cogToFrontAxle': 1.4580434338212815,
 'cogToLeftWheelAxle': 0.8979959585516578,
 'cogToRearAxle': 1.1359565661787185,
 'cogToRightWheelAxle': 0.9020040414483422,
 'distFR': 2.5822227001190186,
 'distLR': 1.8260148763656616,
 'inertia': {'xx': 335.7285923933667,
  'xy': -0.3788520105876442,
  'xz': -15.991984956425028,
  'yy': 1163.27459030952,
  'yz': 0.6530423244225664,
  'zz': 1312.9720537723476},
 'refNodePos': [-6.839996484586663e-05,
  -0.013731973044627921,
  0.29269647276078103],
 'totalMass': 851.1800000000101,
 'vectorForward': [0.0, -1.0, 0.0],
 'vectorLeft': [1.0, 0.0, 0.0],
 'vectorUp': [0.0, 0.0, 1.0],
 'vehHeight': 1.6200000047683716,
 've

In [ ]:
# Let's try to output all these infos in a yaml file
vehicle_config_folder = "../config/vehicles/"
veh_name = "ego"
vehicle_path = vehicle_config_folder + scenario_infos["vehicles_part"][veh_name] + ".yaml"

final_dict_params = {
    **round_dict_values(kin_props),
    "engine" : round_dict_values(engine_infos),
    "gearbox": gear_infos,
    "powertrain": round_dict_values(powertrain_infos),
    "controllers": controllers_infos,
}
save_yaml(
    final_dict_params,
    vehicle_path,
    sort_keys=False
)